# Survival preview

Three-tier baseline decomposition for influence decay. Includes interpretation guardrails.

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path('..').resolve()
for base in (ROOT / 'dataset_v0.2_candidate' / 'metadata', ROOT / 'data' / 'metadata'):
  path = base / 'survival_preview.json'
  if path.exists():
    META = path
    break
else:
  raise FileNotFoundError('survival_preview.json not found')

report = json.loads(META.read_text(encoding='utf-8'))
print('Event rule:', report.get('event_rule'))
print('Warning:', report.get('interpretation_warning'))

In [ ]:
rows = []
for key, comp in report.get('comparisons', {}).items():
  if comp.get('logrank_test') is None:
    continue
  rows.append({
    'comparison': key,
    'hndl_events': comp['events']['hndl'],
    'baseline_events': next(v for k, v in comp['events'].items() if k != 'hndl'),
    'logrank_p': comp['logrank_test']['p_value'],
    'cox_hr_hndl': comp['cox_ph']['hazard_ratio_is_hndl'],
  })

df = pd.DataFrame(rows)
display(df)

decomp = report.get('decomposition', {})
print('Pattern:', decomp.get('pattern_classification'))
print('Interpretation:', decomp.get('interpretation'))

## Guardrails

- Do not equate citation persistence with trade secret value V(t).
- Tier C (fully matched) neutrality suggests influence accumulation, not sector labels, explains differences.
- High censoring; avoid naive median comparisons.